This code is the work of Farokh Cooper (CWID: 20016578)
Parts of this code have been inspired by the Coursera course Special Recommendation Systems as this was a novel approach for me





#Importing headers 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
import os
import glob
from datetime import date
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

- After the epoch of the model is finished, the log is saved to logs to check the training process, and the model per epoch is saved to weights. 

This is a generic piece of code inspired by Coursera course Special Recommendation Systems as it cannot be implemented in another way.

In [ ]:
## Create the logs and weights files if they don't already exist.
## This helps to 

if glob.glob('logs') != ['logs']:
    os.mkdir('logs')
else:
    pass

if glob.glob('weights') != ['weights']:
    os.mkdir('weights')
else:
    pass

In [ ]:
# Set and save the name of the model to be trained in log_model

log_model = 'nmf'       ## saving the model to nmf

today = date.today()
log_formatter = logging.Formatter("%(asctime)s %(message)s")
logger = logging.getLogger()

log_file_name = "./logs/{}_{}".format(today, log_model)       ## adding the log file

file_handler = logging.FileHandler("{}.log".format(log_file_name))
file_handler.setFormatter(log_formatter)
logger.addHandler(file_handler)
logger.setLevel(logging.DEBUG)

#Data Preprocessing

Using the 1 million MovieLens dataset as the larger 20 million dataset is not able to be used to create the NN due to memory running out.

In [ ]:
## Setting the ratings columns

ratings_cols = ['UserID', 'MovieID', 'Rating', 'Timestamp']
ratings = pd.read_csv('ratings.dat', sep='::', engine='python', names=ratings_cols)
ratings

In [ ]:
pd.read_csv('ratings.dat').head()

In [ ]:
ratings['UserID'].isnull().any()

In [ ]:
ratings['MovieID'].isnull().any()

In [ ]:
ratings['Rating'].isnull().any()


In [ ]:
user_ratings_count = ratings.groupby(['UserID']).count()['MovieID']
movie_ratings_count = ratings.groupby(['MovieID']).count()['UserID']
print("Number of unique users in ratings data:", len(user_ratings_count))
print("Number of unique movies in ratings data:", len(movie_ratings_count))

In [ ]:
f, (ax1, ax2) = plt.subplots(1,2)

sns.distplot(user_ratings_count, ax = ax1)
ax1.set_title('Frequency Distribution of Ratings by User')
sns.distplot(movie_ratings_count, ax = ax2)
ax2.set_title('Frequency Distribution of Ratings per Movie')

ax1.set_xlabel('Number of Ratings')
ax2.set_xlabel('Number of Ratings')
plt.subplots_adjust(0.1, 0.1, 2, 1.4)
plt.show()

# Constructing the Model

- Setting the Dataloader and the model settings for implementing PyTorch:



In [ ]:
## Creating a dataset class consisting of users for UserID, items for MovieID, and y for the rating (output).
## Importing dataset and dataloader objects from PyTorch
## This piece of code is inspired by the PyTorch documentation. 

from torch.utils.data import Dataset, TensorDataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, users, items, y):
        self.x = torch.cat([
            torch.LongTensor(users).unsqueeze(0).transpose(0, 1),
            torch.LongTensor(items).unsqueeze(0).transpose(0, 1)
        ], axis=1)
        self.y = torch.FloatTensor(y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.y)

In [ ]:
## Configuring the data train_dataset by labeling users and items from the custom_dataset using LabelEncoder
## This creates a training data loader object.

from sklearn import preprocessing
le1 = preprocessing.LabelEncoder()      ## Label Encoders are used to convert categorical data to numerical data
le2 = preprocessing.LabelEncoder()

batch_size = 256

train_dataset = CustomDataset(          ## Creating our training dataset
    le1.fit_transform(ratings.UserID),
    le2.fit_transform(ratings.MovieID),
    ratings.Rating.values
)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

In [ ]:
## Configureing the model by setting the number of unique users found , the number of movies, and the dimension to embedding.


class MF(nn.Module):                                    ## Creating PyTorch Neural Network module
    def __init__(self, user_ratings_count, movie_ratings_count, emb_size=128):
        super(MF, self).__init__()
        self.user_emb = nn.Embedding(user_ratings_count, emb_size)
        self.item_emb = nn.Embedding(movie_ratings_count, emb_size)
        self.user_emb.weight.data.uniform_(0, 0.05)
        self.item_emb.weight.data.uniform_(0, 0.05)
        
    def forward(self, data):              ## The forward function is defined
        x, y = data[:,:1], data[:,1:]
        u, v = self.user_emb(x), self.item_emb(y)
        return (u.squeeze(1) * v.squeeze(1)).sum(1)   ## The rating is calculated by dot producting the embedding values ??of users and movies.

Generic piece of code inspired by the Coursera course Special Recommendation Systems to check whether our device has a GPU and set the device to GPU if available, else setting it to CPU by default.

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print('The device is: %s' % device)

In [ ]:
model = MF(user_ratings_count, movies_rating_count, emb_size=128).to(device)
print('Constructing the model with:\n\n', model)

In [ ]:
## Setting the optimizer, learning rate and the loss function
learning_rate = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)      ## Selecting the adam optimizer
loss_fn = torch.nn.MSELoss()

In [ ]:
## Training the model here
model.train()

## Setting the number of epochs to be 10, and using the Neural network to train the data
epochs = 10

for e in range(epochs):
    print('start : '+str(e)+' epoch')
    avg_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_hat = model(x)
        loss = loss_fn(y_hat, y)
        avg_loss += loss.item()
        loss.backward()
        optimizer.step()
    torch.save(model.state_dict(), "./weights/{}_{}.pt".format(log_model,e+1))      ## generic piece of code to save the model dictionary obtained in case kernel dies
    print("e{}: loss: {}".format(e+1, loss/(batch_idx+1)))
    logger.info("e{}: loss: {}".format(e+1, loss/(batch_idx+1)))                    ## saving the loss value to the logs too
print('training complete')


# Constructing the model:

In [ ]:
## We evaluate the model again

model.eval()

In [ ]:
## We generate the data to verify our training results 
ratings['le_UserID'] = le1.fit_transform(ratings.UserID)
ratings['le_MovieID'] = le1.fit_transform(ratings.MovieID)

ratings

In [ ]:
## We coonvert the values of our data to LongTensor format as int 64 may not be enough to store, and put it in the model to predict the calculated ratings

val_tensor = torch.LongTensor(ratings[['le_UserID', 'le_MovieID']].values).to(device)
pred = model(torch.LongTensor(ratings[['le_UserID', 'le_MovieID']].values).to(device))

In [ ]:
## Adding the predicted values ??to the ratings Data Frame
ratings['pred_Rating'] = pred.to('cpu').detach().numpy()

# Here we finally obtain our Ratings prediction which can be seen side-by-side with the real ratings.

In [ ]:
ratings